# Aarya Developer Cloud: Colab + Render
This notebook uses Colab as a temporary development/test environment and Render as the hosted FastAPI backend. Run cells in order. Provider API keys stay on Render; only the separate app access token is stored in Colab Secrets.

## 1. Clone the GitHub repository

In [ ]:
from pathlib import Path
import subprocess, sys, platform

REPO_URL = 'https://github.com/Aaryashreyansh/Aarya-Developer-Cloud.git'
REPO_DIR = Path('/content/Aarya-Developer-Cloud')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
print('Repository ready:', REPO_DIR)
print('Python:', sys.version)
print('Platform:', platform.platform())

## 2. Install dependencies and run automated tests

In [ ]:
%cd /content/Aarya-Developer-Cloud
%pip -q install -r requirements.txt
!pytest -q

## 3. Check the Render backend
After creating the Render Blueprint, paste its public HTTPS base URL below. Do not include `/health` or `/docs` at the end.

In [ ]:
import httpx
from getpass import getpass

RENDER_BASE_URL = input('Render HTTPS base URL (for example https://your-service.onrender.com): ').strip().rstrip('/')
if not RENDER_BASE_URL.startswith('https://'):
    raise ValueError('Use the HTTPS URL shown in your Render dashboard.')
with httpx.Client(timeout=60.0) as client:
    health = client.get(f'{RENDER_BASE_URL}/health')
    health.raise_for_status()
    print('Health:', health.json())
    providers = client.get(f'{RENDER_BASE_URL}/api/v1/providers')
    providers.raise_for_status()
    print('Available provider adapters:', [p['id'] for p in providers.json()['providers']])

## 4. Optional authenticated AI smoke test
In Colab's left sidebar open **Secrets** (key icon), add `AARYA_API_TOKEN` with the same value as Render's `APP_API_TOKEN`, and grant this notebook access. Provider API keys and model IDs must already be configured in Render. The app token is never printed.

In [ ]:
from google.colab import userdata

app_token = userdata.get('AARYA_API_TOKEN')
provider = 'openai'  # Change to gemini, claude, or openrouter if configured on Render.
payload = {
    'provider': provider,
    'prompt': 'Reply with exactly: Aarya Developer Cloud connection successful.',
    'system_prompt': 'You are a concise connection-test assistant.'
}
with httpx.Client(timeout=90.0) as client:
    response = client.post(
        f'{RENDER_BASE_URL}/api/v1/chat',
        headers={'X-API-Token': app_token},
        json=payload,
    )
    response.raise_for_status()
    result = response.json()
print('Provider:', result['provider'])
print('AI response:', result['text'])

## Notes
- Colab runtimes are temporary and may disconnect. Render is the hosted backend.
- Do not paste API keys or app tokens into code cells, notebook outputs, GitHub files, or screenshots.
- Never commit generated `.env` files.
- Provider calls can incur charges and are subject to account quotas.
- The backend returns generated text; it does not execute generated code.